# LLM Basics & Tokenisation - Hands-On Tutorial

**Learning Objectives:**
1. Understand transformer architecture at a high level
2. Master tokenization with BPE
3. Work with embeddings for semantic similarity
4. Use Hugging Face pipelines confidently

---

## Setup: Install Required Libraries

In [ ]:
# Install dependencies
!pip install -q transformers tiktoken sentence-transformers torch scikit-learn

---
## Part 1: Tokenization - Token vs Word

**Key Concept:** Tokens are NOT the same as words!

In [ ]:
import tiktoken

# Load GPT-4's tokenizer
enc = tiktoken.encoding_for_model("gpt-4")

# Test cases: token vs word
examples = [
    "Hello",
    "Hello!",
    "supercalifragilisticexpialidocious",
    "ChatGPT",
    "artificial intelligence",
    "I love machine learning!"
]

print("TOKEN vs WORD ANALYSIS")
print("=" * 70)

for text in examples:
    tokens = enc.encode(text)
    word_count = len(text.split())
    token_count = len(tokens)

    print(f"\nText: '{text}'")
    print(f"  Words: {word_count}")
    print(f"  Tokens: {token_count}")
    print(f"  Token IDs: {tokens}")

    # Decode each token
    decoded = [enc.decode([t]) for t in tokens]
    print(f"  Decoded tokens: {decoded}")

### Exercise 1.1: Calculate Token-to-Word Ratio

In [ ]:
# Sample text
sample_text = """
Large Language Models (LLMs) like GPT-4 have revolutionized natural language processing.
They use transformer architecture to understand and generate human-like text.
"""

# Count tokens and words
tokens = enc.encode(sample_text)
words = sample_text.split()

token_count = len(tokens)
word_count = len(words)
ratio = token_count / word_count

print(f"Text: {sample_text.strip()}")
print(f"\nWords: {word_count}")
print(f"Tokens: {token_count}")
print(f"Ratio: {ratio:.2f} tokens per word")
print(f"\nRule of thumb: 1 word ≈ 1.3 tokens (English)")

---
## Part 2: BPE (Byte Pair Encoding) Visualization

**How BPE works:** Iteratively merge the most frequent character pairs

In [ ]:
def simple_bpe_demo(text, num_merges=3):
    """Simple BPE demonstration (educational, not actual implementation)"""

    # Start with characters
    tokens = list(text)
    print(f"Initial: {tokens}")
    print(f"Token count: {len(tokens)}")

    # Simulate merges
    merges = [
        ('i', 'n', 'in'),
        ('in', 'g', 'ing'),
        ('learn', 'ing', 'learning')
    ]

    for i, (a, b, merged) in enumerate(merges[:num_merges], 1):
        # Conceptual merge (simplified)
        print(f"\nMerge {i}: '{a}' + '{b}' → '{merged}'")

    # Show actual tokenization
    actual_tokens = enc.encode(text)
    decoded = [enc.decode([t]) for t in actual_tokens]

    print(f"\nActual GPT-4 tokenization:")
    print(f"  Tokens: {decoded}")
    print(f"  Count: {len(decoded)}")

# Demo
simple_bpe_demo("machine learning")

### Exercise 2.1: Analyze Your Own Text

In [ ]:
# TODO: Try your own text here!
my_text = "Enter your text here"

tokens = enc.encode(my_text)
decoded = [enc.decode([t]) for t in tokens]

print(f"Text: {my_text}")
print(f"Tokens: {decoded}")
print(f"Count: {len(tokens)}")

---
## Part 3: Embeddings - Capturing Meaning

**Key Concept:** Convert text to numeric vectors that capture semantic meaning

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# Load embedding model
model = SentenceTransformer('all-MiniLM-L6-v2')

# Example words
words = ["king", "queen", "man", "woman", "prince", "princess"]

# Create embeddings
embeddings = model.encode(words)

print(f"Embedding shape for '{words[0]}': {embeddings[0].shape}")
print(f"Each word → {embeddings[0].shape[0]} numbers!")
print(f"\nFirst 10 numbers of 'king': {embeddings[0][:10]}")

### Calculate Semantic Similarity

In [ ]:
# Calculate pairwise similarities
print("SEMANTIC SIMILARITY MATRIX")
print("=" * 70)

for i, word1 in enumerate(words):
    for j, word2 in enumerate(words):
        if i < j:  # Only upper triangle
            sim = cosine_similarity(
                embeddings[i].reshape(1, -1),
                embeddings[j].reshape(1, -1)
            )[0][0]

            print(f"{word1:10s} ↔ {word2:10s}: {sim:.3f}")

### The Famous Vector Math: king - man + woman ≈ queen

In [ ]:
# Get individual embeddings
king_emb = embeddings[0]
queen_emb = embeddings[1]
man_emb = embeddings[2]
woman_emb = embeddings[3]

# Vector arithmetic: king - man + woman
result_vector = king_emb - man_emb + woman_emb

# Calculate similarity to each word
print("king - man + woman ≈ ?")
print("=" * 40)

for i, word in enumerate(words):
    sim = cosine_similarity(
        result_vector.reshape(1, -1),
        embeddings[i].reshape(1, -1)
    )[0][0]

    print(f"{word:10s}: {sim:.4f}")

print("\n✓ 'queen' has highest similarity!")

### Exercise 3.1: Test Your Own Analogies

In [ ]:
# TODO: Try your own analogy!
# Example: Paris - France + Italy ≈ Rome

test_words = ["Paris", "France", "Italy", "Rome", "Berlin", "Germany"]
test_embeddings = model.encode(test_words)

# Calculate: Paris - France + Italy
result = test_embeddings[0] - test_embeddings[1] + test_embeddings[2]

print("Paris - France + Italy ≈ ?")
for i, word in enumerate(test_words):
    sim = cosine_similarity(
        result.reshape(1, -1),
        test_embeddings[i].reshape(1, -1)
    )[0][0]
    print(f"{word:10s}: {sim:.4f}")

---
## Part 4: Hugging Face Pipelines - 3 Lines of Magic

**Key Concept:** Use pre-trained models without training

In [ ]:
from transformers import pipeline

# Text Generation
generator = pipeline('text-generation', model='gpt2')

prompt = "The future of artificial intelligence is"
output = generator(prompt, max_length=50, num_return_sequences=1)

print("TEXT GENERATION")
print("=" * 70)
print(f"Prompt: {prompt}")
print(f"\nGenerated: {output[0]['generated_text']}")

### Sentiment Analysis Pipeline

In [ ]:
# Sentiment analysis
sentiment_analyzer = pipeline('sentiment-analysis')

texts = [
    "I love this product! It's amazing!",
    "This is terrible. Very disappointed.",
    "It's okay, nothing special."
]

print("SENTIMENT ANALYSIS")
print("=" * 70)

for text in texts:
    result = sentiment_analyzer(text)[0]
    print(f"\nText: {text}")
    print(f"Sentiment: {result['label']} (confidence: {result['score']:.3f})")

### Question Answering Pipeline

In [ ]:
# Question answering
qa_pipeline = pipeline('question-answering')

context = """
The transformer architecture was introduced in the paper 'Attention is All You Need' in 2017.
It revolutionized natural language processing by using self-attention mechanisms instead of recurrence.
Modern LLMs like GPT-4 and BERT are built on transformer architecture.
"""

questions = [
    "When was the transformer introduced?",
    "What mechanism does it use?",
    "What models use transformers?"
]

print("QUESTION ANSWERING")
print("=" * 70)
print(f"Context: {context.strip()}")

for question in questions:
    result = qa_pipeline(question=question, context=context)
    print(f"\nQ: {question}")
    print(f"A: {result['answer']} (confidence: {result['score']:.3f})")

---
## Part 5: Real-World Application - Semantic Search

**Project:** Build a semantic search engine

In [ ]:
class SemanticSearchEngine:
    def __init__(self, model_name='all-MiniLM-L6-v2'):
        self.model = SentenceTransformer(model_name)
        self.documents = []
        self.embeddings = None

    def add_documents(self, documents):
        """Add documents to search corpus"""
        self.documents = documents
        self.embeddings = self.model.encode(documents)
        print(f"✓ Indexed {len(documents)} documents")

    def search(self, query, top_k=3):
        """Search for most relevant documents"""
        # Encode query
        query_embedding = self.model.encode([query])

        # Calculate similarities
        similarities = cosine_similarity(query_embedding, self.embeddings)[0]

        # Get top-k results
        top_indices = np.argsort(similarities)[::-1][:top_k]

        results = []
        for idx in top_indices:
            results.append({
                'document': self.documents[idx],
                'similarity': similarities[idx],
                'rank': len(results) + 1
            })

        return results

# Demo
search_engine = SemanticSearchEngine()

# Sample documents
documents = [
    "Machine learning is a subset of artificial intelligence",
    "Python is a popular programming language for data science",
    "Deep learning uses neural networks with multiple layers",
    "Natural language processing helps computers understand human language",
    "Transformers revolutionized NLP with attention mechanisms",
    "Data visualization helps communicate insights effectively"
]

search_engine.add_documents(documents)

In [ ]:
# Test searches
queries = [
    "What is AI and neural networks?",
    "Programming for data analysis",
    "Understanding human text with computers"
]

print("SEMANTIC SEARCH RESULTS")
print("=" * 70)

for query in queries:
    print(f"\nQuery: '{query}'")
    print("-" * 70)

    results = search_engine.search(query, top_k=3)

    for result in results:
        print(f"{result['rank']}. {result['document']}")
        print(f"   Similarity: {result['similarity']:.3f}")

### Exercise 5.1: Add Your Own Documents

In [ ]:
# TODO: Add your own documents and test queries!
my_documents = [
    # Add your documents here
]

# Create search engine
# my_engine = SemanticSearchEngine()
# my_engine.add_documents(my_documents)
# results = my_engine.search("your query")

---
## Part 6: Cost Calculation with Token Counting

**Real-world application:** Estimate API costs accurately

In [ ]:
class LLMCostCalculator:
    def __init__(self, model='gpt-4'):
        self.enc = tiktoken.encoding_for_model(model)

        # Pricing (per 1K tokens)
        self.pricing = {
            'gpt-4': {'input': 0.03, 'output': 0.06},
            'gpt-3.5-turbo': {'input': 0.001, 'output': 0.002}
        }
        self.model = model

    def count_tokens(self, text):
        return len(self.enc.encode(text))

    def calculate_cost(self, input_text, output_text):
        input_tokens = self.count_tokens(input_text)
        output_tokens = self.count_tokens(output_text)

        input_cost = (input_tokens / 1000) * self.pricing[self.model]['input']
        output_cost = (output_tokens / 1000) * self.pricing[self.model]['output']

        return {
            'input_tokens': input_tokens,
            'output_tokens': output_tokens,
            'total_tokens': input_tokens + output_tokens,
            'input_cost': input_cost,
            'output_cost': output_cost,
            'total_cost': input_cost + output_cost
        }

# Example usage
calculator = LLMCostCalculator('gpt-4')

# Sample conversation
user_input = "Explain machine learning in simple terms"
assistant_output = "Machine learning is a way for computers to learn from data without being explicitly programmed. It uses algorithms to find patterns and make predictions."

cost_breakdown = calculator.calculate_cost(user_input, assistant_output)

print("COST BREAKDOWN")
print("=" * 70)
print(f"Model: {calculator.model}")
print(f"\nInput tokens: {cost_breakdown['input_tokens']}")
print(f"Output tokens: {cost_breakdown['output_tokens']}")
print(f"Total tokens: {cost_breakdown['total_tokens']}")
print(f"\nInput cost: ${cost_breakdown['input_cost']:.6f}")
print(f"Output cost: ${cost_breakdown['output_cost']:.6f}")
print(f"Total cost: ${cost_breakdown['total_cost']:.6f}")

### Scale to Production

In [ ]:
# Estimate costs at scale
conversations_per_day = 10_000
days_per_month = 30

daily_cost = cost_breakdown['total_cost'] * conversations_per_day
monthly_cost = daily_cost * days_per_month

print("PRODUCTION COST ESTIMATE")
print("=" * 70)
print(f"Cost per conversation: ${cost_breakdown['total_cost']:.4f}")
print(f"Conversations per day: {conversations_per_day:,}")
print(f"\nDaily cost: ${daily_cost:,.2f}")
print(f"Monthly cost: ${monthly_cost:,.2f}")
print(f"Annual cost: ${monthly_cost * 12:,.2f}")

---
## Summary & Key Takeaways

**1. Transformers:**
- Architecture behind all modern LLMs
- Encoder (understand) + Decoder (generate)
- GPT = decoder-only, BERT = encoder-only

**2. Tokenization:**
- Tokens ≠ Words (average: 1.3 tokens/word)
- BPE breaks words into sub-word units
- Critical for API cost calculation

**3. Embeddings:**
- Text → numeric vectors (384-1536 dimensions)
- Captures semantic meaning
- Enables: similarity search, analogies, clustering

**4. Hugging Face:**
- Pre-trained models ready to use
- Pipeline API = 3 lines of code
- Tasks: generation, classification, QA, etc.

---
## Next Steps

1. Experiment with different models on Hugging Face Hub
2. Build your own semantic search application
3. Try fine-tuning a model for your specific task
4. Explore advanced tokenization techniques

**Happy Learning! 🚀**